## NLP with the corpus of African youth literature

### Introduction

This notebook demonstrates the use of NLP tools with a corpus of 155 texts from the African youth literature genre using Python code. The notebook was created as a supportive guide to the quantitative data presented in the dissertation. The notebook covers *topic modelling* and retrieving *word frequencies* and *collocations*. The results are presented and analysed in more detail in Chapter 5 of the dissertation. 

All the files required to run this program are available in a GitHub repository at: https://github.com/BronwynBowlesKing/African-youth-literature-corpus-analysis. This includes the corpus plain text files ('jaylit_african_youth_literature_corpus') and a term filter list ('term_filter_list_pos.csv').

The process starts with preparing and cleaning the texts, transforming them into a suitable format, and then applying Latent Dirichlet Allocation (LDA) for topic modelling to identify latent themes (Blei et al., 2003). The notebook also includes methods for evaluating topic coherence using semantic similarity based on an English spaCy module.

The following topic modelling workflow is applied:

* Installation and loading of packages (Step 0.1 and 0.2)
  
* Definition of core functions (0.3)

* Loading and previewing the corpus (0.4)

* Text cleaning (1.1)

* Tokenisation (1.2)

* Removal of punctuation and numbers (1.3)
  
* Lemmatisation (1.4)
  
* Remove stopwords (1.5)
  
* Remove filter terms (1.6)

* Construction of the document-term matrix (DTM) (2.1)

* Baseline topic modelling test (2.2)
  
* Topic coherence evaluation (2.3)
  
* Selection of the final topic model (2.4)

Lists of frequent terms and word counts are obtained as the corpus is cleaned and prepared for analysis. The frequency of any word in the corpus can be found using the code in Step 3, with "eye" being the focus here, in line with the research aims and questions. To aid the research further, the context windows (collocations) of a target word in the original corpus are returned using the code in Step 4. Again, "eye" is targeted in this step.

### Step 0. Preparation

This step covers the initial setup required before the main processes for text cleaning and topic modelling can begin. It includes installing and loading the necessary Python libraries with the tools for text processing. 

#### 0.1 Install packages and modules

If needed, these packages and modules that are not part of the default Python library can be installed or downloaded. It is recommended to install packages in the terminal using the following commands, rather than in the notebook.

In [1]:
# pip install numpy pandas matplotlib seaborn scikit-learn nltk spacy
# python -m spacy download en_core_web_sm
# python -m spacy download en_core_web_lg

#### 0.2 Load packages and modules

In [2]:
import os
import re
import numpy as np
import pandas as pd
from itertools import combinations
from collections import Counter
import spacy
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.decomposition import LatentDirichletAllocation

In [3]:
# Load spaCy modules and disable unused pipeline components for faster processing
# For lemmatisation
nlp_sm = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

# For coherence scores
nlp_lg = spacy.load('en_core_web_lg', disable=['parser', 'ner', 'tagger'])  

#### 0.3 Define functions

Functions are defined below for cleaning the text, counting the number of words in a group of texts, calculating topic coherence scores, working with word embeddings to measure semantic similarity, and finding word collocations. The last function allows neater display of tables with headings. Each function is defined in more detail in a docstring. 

In [4]:
def clean_text(text):
    '''
    Normalises whitespace with a regex pattern. Converts different types 
    of line breaks and tabs into single spaces, preparing the text for 
    tokenisation and further processing. 
    '''    

    text = re.sub(r'[\n\r\t]', ' ', text) 
    return text


def word_counter(df, id_list):
    '''
    To count words in groups of texts based on a given list of doc ids.
    '''
    
    subset = df[df['doc_id'].isin(id_list)]
    return subset['cleaned_text'].str.split().str.len().sum()


def get_embedding(word):
    '''
    Helper (secondary) function to the main function 'topic_coherence'.
    Obtains numerical embeddings (vector representations) for given words 
    (tokens) using a pre-trained spaCy language model. For NLP purposes, 
    where word meaning needs to be represented mathematically. 
    '''

    token = nlp_lg.vocab[word]
    if token.has_vector:
        return token.vector
    else:
        return None


def cosine_similarity(vec1, vec2):
    '''
    Helper function to the main function 'topic_coherence'. Computes the 
    cosine similarity between two vectors.
    '''

    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 > 0 and norm2 > 0:
        return dot_product / (norm1 * norm2)
    else:
        return 0.0


def topic_coherence(top_terms):
    '''
    Computes average pairwise cosine similarity between spaCy embeddings 
    of top terms in topics derived from topic modelling. Provides a measure 
    of semantic coherence (conceptual similarity) for topics discovered by 
    a topic modelling algorithm. Depends on helper functions get_embedding 
    and cosine_similarity. 
    
    Unique pairs of vectors are compared for topic terms (cosine_similarity(vec1, vec2)) 
    to measure how similar they are on a scale from -1 to 1, and the mean similarity 
    score is returned (np.mean(sim_scores)).

    For error-handling, in an unlikely case where fewer than two words in a topic 
    have available embeddings (len(vectors) < 2), the function cannot compute 
    similarity, and thus None is returned.
    '''

    vectors = []
    # Extract embeddings for each term
    for term in top_terms:
        vec = get_embedding(term)
        if vec is not None:
            vectors.append(vec)

    if len(vectors) < 2:
        # If fewer than 2 terms have vectors, coherence is not computed
        return np.nan
    
    sim_scores = []
    # Compute pairwise cosine similarity among all vectors
    for vec1, vec2 in combinations(vectors, 2):
        sim = cosine_similarity(vec1, vec2)
        sim_scores.append(sim)

    return np.mean(sim_scores) if sim_scores else np.nan


def find_collocations(cleaned_texts, target_words, window_size=6):
    '''
    Search for all occurrences of a target word in a set of texts and return the 
    context windows. Returns all examples regardless of case, but searches for 
    whole words only. 
    
    target_words: str, one or more words to search for
    window_size: int, number of words before and after to return
    '''

    results = []
    texts_with_matches = set()
    total_matches = 0
    
    for entry in cleaned_texts:
        text = entry['cleaned_text']
        doc_id = entry['doc_id']
        found_in_doc = False
        
        # Tokenise while preserving word position
        # Apply regex to split on whitespace and keep punctuation with words
        words = re.findall(r'\S+', text)
        
        # Check each word to see if it matches target word (case insensitive)
        for i, word in enumerate(words):
            # Normalise word for comparison (lowercase, remove punctuation)
            normalised_word = re.sub(r'[^\w]', '', word.lower())
            
            if any(target.lower() == normalised_word for target in target_words):
                # Retrieve context window
                start = max(0, i - window_size)
                end = min(len(words), i + window_size + 1)
                
                context = words[start:end]
                target_pos = i - start
                
                results.append({
                    'doc_id': doc_id,
                    'target_word': word,
                    'context': ' '.join(context),
                    'target_position': target_pos
                })
                
                found_in_doc = True
                total_matches += 1
        
        if found_in_doc:
            texts_with_matches.add(doc_id)
    
    # Print diagnostic information
    print(f'Total matches of target word/s found: {total_matches}')
    print(f'Number of texts with matches: {len(texts_with_matches)} out of {len(cleaned_texts)}')
    print(f'Number of texts without matches: {len(cleaned_texts) - len(texts_with_matches)}')
    
    return pd.DataFrame(results)


def display_table(df, caption, color='#000000D3'):
    '''
    Displays key dataframes (dfs) as styled tables with headings. Hides index col. 
    Numbers are shown to four decimal places.

    Parameters:
    - df (pd.DataFrame): df to display
    - caption (str): Header text above the table
    '''
    
    styles = [dict(
        selector='caption', props=[('font-weight', 'bold'), ('color', color)]
    )]
    
    display(
        df.style
            .format(precision=4)
            .hide(axis='index')
            .set_caption(caption)
            .set_table_styles(styles)
    )

#### 0.4 Set the file directory, load and preview the plain text files

The African youth literature corpus is loaded into memory and previewed with the code below. The corpus texts are available in the GitHub repository. Each file is added to a Python list with document IDs for the text based on the order in which the texts were originally published.

In [ ]:
# Specify file directory
file_dir = 'jaylit_african_youth_literature_corpus'  

# List all text files 
files = [f for f in os.listdir(file_dir) if f.endswith('.txt')]

# Load contents of each text file into list
texts = []
for file_name in files:
    if file_name.endswith('.txt'):
        # Extract doc ids from file name
        doc_id = file_name.split(' ')[0]  
        with open(
            os.path.join(file_dir, file_name), 'r', encoding='utf-8'
        ) as file:
            text = file.read()
            texts.append({'doc_id': doc_id, 'text': text})

# Verify the number of files processed. Should be 155 
num_files = len(files)
print(f'Number of files processed: {num_files}')

Number of files processed: 155


### Step 1. Text pre-processing

#### 1.1 Text cleaning

The cleaning function is called here to normalise whitespace and a dataframe of cleaned texts is created. The number of words is counted and displayed for the whole corpus and then for each genre.

In [6]:
cleaned_texts = []

for entry in texts:  
    cleaned_text = clean_text(entry['text'])   
    cleaned_texts.append(
        {'doc_id': entry['doc_id'], 'cleaned_text': cleaned_text}
    )

cleaned_texts_df = pd.DataFrame(cleaned_texts)
cleaned_texts_df['doc_id'] = cleaned_texts_df['doc_id'].astype(int)  # doc_ids to int format

# Count the total number of words
word_count = cleaned_texts_df['cleaned_text'].str.split().str.len().sum()
print(f'Total corpus word count: {word_count}')

# Specify text ids by genre
poetry_ids = [1, 2, 3, 9, 10, 11, 12, 13, 14, 15, 16, 
            17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 
            27, 28, 29, 30, 31, 32, 33, 49, 54, 55, 
            56, 57, 58, 59, 60, 61, 62, 63, 64, 65,
            66, 67, 68, 69, 70, 71, 72, 73, 79, 80, 
            81, 82, 83, 84, 85, 99, 100, 101, 102, 
            103, 104, 105, 106, 107, 115, 116, 117, 
            118, 119, 120, 121, 122, 123, 124, 125, 
            126, 127, 128, 129, 130, 131, 132, 133, 
            134, 135, 136, 137, 138, 139, 149, 150, 
            151, 152, 153, 154, 155, 156, 157, 158, 
            172, 173, 174, 175, 176, 177, 178, 179, 
            180, 181, 182, 183, 184, 185, 186, 187, 188]
story_ids = [34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 
             44, 91, 92, 140, 141, 142, 143, 144, 159, 
             160, 161, 162, 163, 164, 165, 166, 189, 
             190, 191, 192, 193]
play_ids = [48, 93, 94, 145, 146, 168, 194]

# Count the total number of words by genre with defined function 
poetry_word_count = word_counter(cleaned_texts_df, poetry_ids)
story_word_count = word_counter(cleaned_texts_df, story_ids)
play_word_count  = word_counter(cleaned_texts_df, play_ids)

print(f'Poetry word count: {poetry_word_count}')
print(f'Short stories word count: {story_word_count}')
print(f'Plays word count: {play_word_count}')

Total corpus word count: 167922
Poetry word count: 20538
Short stories word count: 115702
Plays word count: 31682


#### 1.2 Tokenisation

Cleaned texts are split into word tokens using NLTK and the results are stored with the doc_id that each token belongs to. A dataframe of then created for further processing (tokens_df_1). This dataframe contains all words and punctuation in the original texts.

In [7]:
# Tokenise cleaned texts
tokens = []
for entry in cleaned_texts:
    tokenised_words = word_tokenize(entry['cleaned_text'].lower())
    for word in tokenised_words:
        tokens.append({'doc_id': entry['doc_id'], 'word': word})
        
# Create df for the tokens 
tokens_df_1 = pd.DataFrame(tokens, columns=['doc_id', 'word'])

#### 1.3 Remove punctuation and numbers

The tokens are filtered to keep only alphabetic strings, and these words are then saved as tokens_df_2. The original terms across all texts are saved in a CSV ('original_term_frequencies.csv'). This includes all words ranked by frequency with no stopwords removed, but digits and punctuation have been removed. The 10 most frequent terms are shown in Table 1. 

In [8]:
# Remove punctuation and numbers and lowercase all words
tokens_df_1['word'] = tokens_df_1['word'].str.replace(r'[^a-zA-Z]', '', regex=True)
tokens_df_2 = tokens_df_1[tokens_df_1['word'].str.isalpha()]
tokens_df_2.loc[:, 'word'] = tokens_df_2['word'].str.lower()
tokens_df_2 = tokens_df_2[tokens_df_2['word'].str.len() > 1]

# Preview and save table of terms and their frequency from original texts
term_frequency_initial = tokens_df_2['word'].value_counts().reset_index()
term_frequency_initial.columns = ['word', 'frequency']

display_table(term_frequency_initial.head(10), 
            'Table 1: Original word frequencies (first 10 rows)')

term_frequency_initial.to_csv('original_term_frequencies.csv', index=False)

word,frequency
the,7737
to,4692
and,4323
of,2950
you,2464
was,2448
she,2323
in,2262
her,2160
it,2061


#### 1.4 Lemmatisation

Terms are now lemmatised to their root form with the spaCy pipeline and saved to the dataframe tokens_lemmatised_df, which will be used in following steps. They are not stemmed as stemming returns terms that do not take into account words like irregular verbs (e.g. run and ran), leading to results that are difficult to interpret. Word frequencies for the top 10 terms after lemmatisation are shown in Table 2.

In [9]:
# Filter out empty strings
words = [w for w in tokens_df_2['word'].tolist() if w] 

# Process words in batches
docs = nlp_sm.pipe(words, batch_size=1000, n_process=1)

# Extract lemmatised forms
lemmas = []
for doc in docs:
    if doc and len(doc) > 0:
        lemmas.append(doc[0].lemma_)
    else:
        lemmas.append('')

tokens_cleaned = tokens_df_2.copy()
tokens_cleaned['word'] = tokens_cleaned['word'].astype(str)

# Remove empty rows
tokens_cleaned = tokens_cleaned[~tokens_cleaned['word'].str.strip().eq('')]  

tokens_lemmatised_df = tokens_cleaned.copy()
tokens_lemmatised_df['word'] = lemmas

# Preview and save table of lemmas and their frequency
term_frequency_lemma = tokens_lemmatised_df['word'].value_counts().reset_index()
term_frequency_lemma.columns = ['word', 'count']

display_table(term_frequency_lemma.head(10), 
            'Table 2: Word frequencies after lemmatisation (first 10 rows)')

term_frequency_lemma.to_csv('term_frequencies_lemmatised.csv', index=False)

word,count
the,7737
be,6203
to,4692
she,4483
and,4323
of,2950
you,2468
in,2262
he,2214
it,2061


#### 1.5 Remove stopwords and retain keep words

A goal of the quantitative component of the research is to get a sense of the overall thematic structure of the corpus. Various iterations of topic modelling can and have been run for the corpus with and without lemmatising and removing filter terms and stopwords to this end. The *keep_words* are a list of words to retain by removing them from the stopwords that will be used in this case. A diction of sight is key to this corpus as "eyes" and related terms are highly frequent words. Thus, "see" and its derivatives are retained.

After removing stopwords, a new dataframe is created (tokens_df_3). Table 3 below shows the top 10 word frequencies from this dataframe. The stopwords are derived from the NLTK package and include a fairly limited list of common words. These words are saved below in a file 'nltk_stopwords.csv' for inspection.

In [10]:
# Load NLTK stopwords and specify keep words
nltk_stopwords = set(stopwords.words('english'))
keep_words = {'see', 'seeing', 'saw', 'seen'}

# Remove stopwords and retain the keep_words
tokens_df_3 = tokens_lemmatised_df[
    (~tokens_lemmatised_df['word'].isin(nltk_stopwords) | tokens_lemmatised_df['word'].isin(keep_words))
]

# Display and save term frequencies
stopwords_removed_df = (
    tokens_df_3['word']
    .value_counts()
    .reset_index()
)

display_table(stopwords_removed_df.head(10),
            'Table 3: Word frequencies – Stopwords removed, keep words retained (first 10 rows)')

# Save word frequencies
stopwords_removed_df.to_csv('stopwords_removed_df.csv', index=False)

# Convert NLTK stopwords to df and save to csv for inspection
df_stopwords = pd.Series(list(nltk_stopwords))
df_stopwords.to_csv('nltk_stopwords.csv', index=False, header=False)

word,count
I,1266
say,828
like,775
go,674
know,667
look,581
see,579
would,524
one,522
get,511


#### 1.6 Remove filter terms

In topic modelling and other computational methods, researchers may remove basic stopwords as well as additional words that do not help identify coherent topics. It has become standard practice to extend stopwords to other corpus- or domain-specific high-frequency words that do not convey meaning relevant to the investigation (e.g., Bystrov et al., 2025; Hardy, 2004; Schofield et al., 2017). 

As the NLTK stopwords list is quite limited, a term filter list (term_filter_list_pos) is applied below to improve topic interpretability. It includes additional stopwords or common words in the corpus that do not indicate theme or topic and skew the results towards words such as character names and basic verbs. The term filter list is available in the GitHub repository. The term filter list includes a column with parts-of-speech (POS) tags for each word and explanations as to why terms are included in the list. 

The main reason for removing additional stopwords and filter terms is that otherwise the topics tend to show poor interpretability or relevance to the research (Van Kessel, 2019). After removing these words below, a new dataframe is created (tokens_df_4), which will be used in the topic modelling steps. Table 4 shows the top 10 word frequencies after stopwords and filter terms are removed.

In [ ]:
# Load term filter list and tokens
filter_list_df = pd.read_csv('term_filter_list_pos.csv')

# Convert to Python list
filter_list = filter_list_df['word'].tolist()

# Remove filter terms from tokens
tokens_df_4 = tokens_df_3[~tokens_df_3['word'].isin(filter_list)]

# Show and save top 10 term frequencies
term_frequency = (
    tokens_df_4['word']
    .value_counts()
    .reset_index()
)

display_table(term_frequency.head(10),
            'Table 4: Word frequencies after stopwords and filter terms removed (first 10 rows)')

term_frequency.to_csv(
    'term_frequencies_stopwords_filter_removed.csv', index=False
)

word,count
say,828
know,667
look,581
see,579
time,399
think,392
feel,358
man,356
tell,335
ask,329


### Step 2. Topic modelling

#### 2.1 Create a DTM

A DTM is now created from tokens_df_4, as this is the correct format for the LDA program to read and work with (Bystrov et al., 2025). Sparse rows are removed to speed up the process and stabilise the model.

In [12]:
dtm = tokens_df_4.groupby(['doc_id', 'word']).size().unstack(fill_value=0)
min_df = 2
dtm = dtm.loc[:, (dtm > 0).sum(axis=0) >= min_df]

#### 2.2 Set the number of topics and run a baseline test

The LDA model can now be initialised and run. The LDA module from scikit-learn (2025) provides a statistical model designed to detect thematic structures in a corpus. It categorises words into topics by estimating which words frequently co-occur, allowing us to identify and interpret major themes present in large text datasets. 

The code below starts with 10 topics as a baseline test. The number of topics is the *k* value. Different *k* values will be tested later by calculating coherence scores. After the topics are derived, they are displayed in Table 5 and saved to a file.

In [13]:
k = 10  # Number of topics to request
n_top_words = 10  # Number of terms per topic

# Fit and test model
lda = LatentDirichletAllocation(n_components=k, random_state=42)
lda.fit(dtm)

# Capture topic-word distributions
topic_word_distributions = lda.components_
feature_names = dtm.columns

top_terms_per_topic = {}

for topic_idx, topic in enumerate(lda.components_):
    top_indices = np.argsort(topic)[::-1][:n_top_words]
    top_terms = [feature_names[i] for i in top_indices]
    # Assign to dictionary with keys 'topic_1', 'topic_2'
    top_terms_per_topic[f'topic_{topic_idx + 1}'] = top_terms

# Convert dictionary to df 
# Columns are topics and rows are terms ranked by importance
topics_10_df = pd.DataFrame(top_terms_per_topic)

# View and save table of topics
display_table(topics_10_df, 'Table 5: Topic modelling results – 10 topics')

topics_10_df.to_csv('lda_10_topics.csv', index=False)

topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,topic_10
tell,king,woman,say,see,say,life,see,say,time
heaven,queen,love,know,man,love,home,know,man,life
love,see,tear,look,look,mother,love,eye,see,know
mama,say,feel,ask,woman,know,say,look,boy,say
letter,palace,life,think,eye,look,search,hand,tell,cry
god,man,cry,see,car,time,hope,face,know,son
heart,time,blue,feel,hand,think,culture,feel,look,village
life,think,hand,smile,walk,mom,feel,away,girl,man
dream,know,girl,time,face,father,strength,mother,people,child
show,eye,voice,face,body,man,african,light,time,think


#### 2.3 Calculate coherence scores for a range of topics

The code below now tests the coherence of different numbers of topics. The coherence scores here are a measure of how often terms co-occur based on existing language models, justifying their being grouped together in a topic. In other words, coherence scores "measure how interpretable the topics are to humans" (Zvornicanin, 2025). A spaCy (2025) language model is used for this purpose. 

The code will iterate through a range of *k* values (from 6 to 20 in increments of 2) and determine the coherence scores. A higher score (closer to 1) indicates that the terms across a certain number of topics are more semantically similar, so that the topic is more likely to be 'coherent'. A lower score (close to 0 or negative) means the words are less related, making the topics not as semantically clear (Zvornicanin, 2025). 

However, the question of coherence is also relative to a particular corpus. While a higher score is generally preferable to a much lower one, the researcher must use their judgment and take practical constraints into account. It is not possible to achieve a 'perfect' score (1). Very high scores can indicate anomalies with the model or the corpus texts, as though they were lists of synonyms. The average expected scores for coherent topics are around 0.4 to 0.5 for most texts (Ajinaja et al., 2022; Murel, 2024). 

Zvornicanin (2025) explains: "The idea behind this method is that we want to choose a point after which the diminishing increase of coherence score is no longer worth the additional increase of the number of topics." The coherence scores are thus not necessarily intended to show the 'correct' number of topics to choose to the exclusion of others that are 'incorrect'. Instead, they show how semantically related the words are in sets of topics and are used as a guide alongside the researcher's judgment.

In [14]:
k_start = 6 # inclusive lower-bound
k_end = 21  # exclusive upper-bound
step = 2
n_top_words = 10  

feature_names = dtm.columns
coherence_scores_by_k = []

for k in range(k_start, k_end, step):
    lda = LatentDirichletAllocation(n_components=k, random_state=42)
    lda.fit(dtm)
    
    topic_coherences = []
    for topic_idx, topic_weights in enumerate(lda.components_):
        top_indices = np.argsort(topic_weights)[::-1][:n_top_words]
        top_terms = [feature_names[i] for i in top_indices]
        cohesion = topic_coherence(top_terms)
        topic_coherences.append(cohesion)
    
    avg_coherence = np.nanmean(topic_coherences)
    coherence_scores_by_k.append(avg_coherence)
    print(f'k = {k} coherence score: {avg_coherence:.3f}')

k = 6 coherence score: 0.518
k = 8 coherence score: 0.457
k = 10 coherence score: 0.453
k = 12 coherence score: 0.456
k = 14 coherence score: 0.449
k = 16 coherence score: 0.425
k = 18 coherence score: 0.419
k = 20 coherence score: 0.397


#### 2.4 Select the optimum number of topics

Except for 20 topics, which is slightly low (0.397), all the topic numbers tested are within an acceptable coherence score range (~0.4 to ~0.5). The differences between the scores for various topics are relatively small, suggesting model stability. These types of results are to be expected when working with a natural language corpus. They support the conclusion that most of these topics are generally acceptable for analysis purposes (Zvornicanin, 2025).
 
However, coherence scores are not the only factor to consider. Although the lower numbers of topics have slightly higher coherence scores, there are practical and interpretative reasons to avoid them. With 6 and 8 topics, I found the topic scope too narrow or 'undercooked', so that the topics do not capture enough diversity and detail for sufficiently meaningful analysis. With 12 to 20 topics, increasing term repetition starts to become apparent. This makes the topics more difficult to interpret and less practically useful.

Therefore, the 10 topics from the baseline test (step 2.2) are an acceptable compromise between these. The coherence score (0.453) for 10 topics is also within the acceptable range, and these topics are not too many to present or too few to capture the diversity of the corpus under study. This selection balances semantic coherence with interpretability and practical analysis needs, which is a best practice in topic modelling (Murel, 2024).

In conclusion to the topic modelling section of this notebook, 10 topics are selected and presented in the dissertation in Chapter 5.

### Step 3. Count occurrences of "eye*"

Noting that "eye*" appears frequently in the corpus and the topic modelling results, this term and its derivatives are counted below in the original tokenised words (tokens_df_1) and then saved as a table.

In [15]:
token_counts = Counter(tokens_df_1['word'])

target_words = [
    'eyes', 'eye', 'eyebrows', 'eyed', 'eyebrow',
    'eyeing', 'eyelid', 'eyelids', 'eyesight', 'eyelashes'
]

word_counts = {word: token_counts.get(word, 0) for word in target_words}

eye_word_counts = pd.DataFrame(word_counts.items(), columns=['word', 'count'])

eye_word_counts_sorted = eye_word_counts.sort_values(by='count', ascending=False).reset_index(drop=True)
total_count = eye_word_counts_sorted['count'].sum()
total_row = pd.DataFrame({
    'word': ['lemma "eye" total'],
    'count': [total_count]
})

eye_word_counts_final = pd.concat([eye_word_counts_sorted, total_row], ignore_index=True)

display_table(eye_word_counts_final, 'Table 7: Frequency of lemma "eye*" in the corpus')

eye_word_counts_final.to_csv('eye_word_counts.csv', index=False)

word,count
eyes,275
eye,28
eyebrows,8
eyebrow,3
eyed,2
eyeing,2
eyelid,2
eyelids,2
eyesight,2
eyelashes,1


### Step 4. Find collocations of "eye" and "eyes"

Collocations of the words "eye" and "eyes" are found and saved using the code below. cleaned_text is used for this purpose as it has normalised whitespace but contains all the words in the text before any further changes were made, such as removing stopwords, lowercasing all text, etc. 

The code below searches for the whole words specified, returning all instances regardless of case. A window of six words before and after the target word is returned, as this was found to give sufficient information on the target word's context. A sample of five context windows is shown, and all the results are then saved to a CSV. The target words can be replaced with other words to find their context windows.

In [16]:
collocations_df = find_collocations(cleaned_texts, ['eye', 'eyes'], window_size=6)

print('\nSample of context windows containing target word/s:\n')
for i, row in collocations_df.head().iterrows():
    print(f'doc_id: {row['doc_id']} | Target: {row['target_word']} | Context: {row['context']}')

collocations_df.to_csv('eye_eyes_collocations.csv', index=False)

Total matches of target word/s found: 303
Number of texts with matches: 75 out of 155
Number of texts without matches: 80

Sample of context windows containing target word/s:

doc_id: 1 | Target: eyes. | Context: is a blurry situation in your eyes. We’ve awoken a beast. We’re not
doc_id: 104 | Target: eyes | Context: to protect And not the shameful eyes to neglect Don't question my sanity
doc_id: 106 | Target: eyes | Context: Do you remember How teasing your eyes were When we kissed That night?
doc_id: 121 | Target: eyes | Context: you hide away in when your eyes are open the ones whose doors
doc_id: 125 | Target: eyes | Context: scars. For there are days When eyes will disagree with The truths of


### Sources

Ajinaja, M. O., Adetunmbi, A. O., Ugwu, C. C. & Popoola, O. S. 2022. Semantic similarity measure for topic modeling using latent Dirichlet allocation and collapsed Gibbs sampling. *Iran Journal of Computer Science*, 6(1):81–94. https://doi.org/10.1007/s42044-022-00124-7

Blei, D. M., Ng, A. Y. & Jordan, M. I. 2003. Latent Dirichlet allocation. *Journal of Machine Learning Research*, 3(1):993–1022.

Bystrov, V., Naboka-Krell, V., Staszewska-Bystrova, A. & Winker, P. 2025. Analysing the impact of removing infrequent words on topic quality in LDA models. *Central European Journal of Economic Modelling and Econometrics*, 25(79):61–85. https://doi.org/10.24425/cejeme.2025.155564

Hardy, D.E. 2004. Collocational Analysis as a Stylistic Discovery Procedure: The Case of Flannery O’Connor’s *Eyes. Style*, 38(4):410–427.

Jockers, M. L. 2013. *'Secret' Recipe for Topic Modeling Themes*. https://www.matthewjockers.net/2013/04/12/secret-recipe-for-topic-modeling-themes Date of access: 13 August 2025.

Murel, J. 2024. *Train an LDA topic model for text analysis in Python*. https://developer.ibm.com/tutorials/awb-lda-topic-modeling-text-analysis-python Date of access: 9 July 2025.

NLTK. 2025a. Sample usage for corpus. https://www.nltk.org/howto/corpus.html Date of access: 8 June 2025.

NLTK. 2025b. nltk.tokenize package. https://www.nltk.org/api/nltk.tokenize.html Date of access: 8 June 2025.

O'Sullivan, J. 2024. collocates. GitHub repository. https://github.com/jamesosullivan/collocates/tree/main

Schofield, A., Magnusson, M. & Mimno, D. 2017. *Pulling out the stops: Rethinking stopword removal for topic models*. Paper presented at the 15th Conference of the European Chapter of the Association for Computational Linguistics, Valencia. https://doi.org/10.18653/v1/E17-2069

scikit-learn. 2025. *LatentDirichletAllocation*. https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html Date of access: 8 June 2025.

spaCy. 2025. *Available trained pipelines for English*. https://spacy.io/models/en Date of access: 8 June 2025.

Van Kessel, P. 2019. *Overcoming the limitations of topic models with a semi-supervised approach*. Pew Research Centre. https://www.pewresearch.org/decoded/2019/04/10/overcoming-the-limitations-of-topic-models-with-a-semi-supervised-approach Date of access: 12 June 2025.

Zvornicanin, E. 2025. *When Coherence Score Is Good or Bad in Topic Modeling.* https://www.baeldung.com/cs/topic-modeling-coherence-score Date of access: 9 June 2025.